# Evaluation: 2021 6-hourly test set from yearly prediction Zarr files

This notebook matches the final evaluation strategy agreed with Simon: run inference for every 6 hours in the 2021 test year (`00:00`, `06:00`, `12:00`, `18:00` each day), save model forecasts as one yearly Zarr store per model, then compute and plot metrics from those stores.

The notebook no longer selects 30 rainy events and no longer runs model inference live.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_RESULTS = Path('/data/brussel/114/vsc11442/code/mlcast-ldcast/results/evaluation/test_2021_6hourly')
EVENTS_CSV = BASE_RESULTS / 'events_2021_6hourly.csv'
METRICS_DIR = BASE_RESULTS / 'metrics'
FIG_DIR = BASE_RESULTS / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SCRATCH = os.environ.get('VSC_SCRATCH', '/scratch/brussel/114/vsc11442')
PRED_BASE = Path(SCRATCH) / 'mlcast_predictions/test_2021_6hourly'
OBS_ROOT = PRED_BASE / 'observations'
PRED_ROOT = PRED_BASE / 'predictions'

MODELS = ['convgru', 'convgru_ens', 'unet', 'unet_ens', 'pysteps', 'ldcast']
MODEL_LABELS = {
    'convgru': 'ConvGRU',
    'convgru_ens': 'ConvGRU ensemble',
    'unet': 'U-Net',
    'unet_ens': 'U-Net ensemble',
    'pysteps': 'PySTEPS',
    'ldcast': 'LDCast',
}
THRESHOLDS_MM_H = [0.1, 0.5, 1.0, 5.0]

print('Prediction base:', PRED_BASE)
print('Metrics dir:', METRICS_DIR)


## Event list check

The expected number of 6-hourly starts in non-leap-year 2021 is `365 × 4 = 1460`.

In [ ]:
if EVENTS_CSV.exists():
    events = pd.read_csv(EVENTS_CSV)
    display(events.head())
    display(events.tail())
    print('Number of events:', len(events))
    print('Months:', sorted(events['month'].unique()))
    print('Hours:', sorted(events['hour'].unique()))
else:
    print('Missing event CSV. Run jobs/build_2021_6hour_events.slurm first.')

## Zarr export inventory

This checks which yearly Zarr stores are available for observations and each model. Missing stores must be generated before final metrics are complete.


In [ ]:
def zarr_inventory():
    rows = []
    obs_path = OBS_ROOT / 'observations_2021.zarr'
    rows.append({'kind': 'observations', 'model': 'observations', 'path': str(obs_path), 'exists': obs_path.exists()})
    for model in MODELS:
        pred_path = PRED_ROOT / model / f'{model}_full_2021.zarr'
        rows.append({'kind': 'prediction', 'model': model, 'path': str(pred_path), 'exists': pred_path.exists()})
    return pd.DataFrame(rows)

inventory = zarr_inventory()
display(inventory)
print('Available stores:', int(inventory['exists'].sum()), '/', len(inventory))


## Load metric CSV files

The metric CSV files are produced by `compute_metrics_from_prediction_zarr.py` in yearly mode. The script computes deterministic error metrics, contingency-table scores, CRPS/Brier/reliability/rank histograms, FSS, and rain-rate histograms from the yearly Zarr forecasts.


In [ ]:
def read_csv_or_empty(name):
    path = METRICS_DIR / name
    if not path.exists():
        print('Missing:', path)
        return pd.DataFrame()
    df = pd.read_csv(path)
    if 'model' in df.columns:
        df['model_label'] = df['model'].map(MODEL_LABELS).fillna(df['model'])
    print(name, 'rows=', len(df))
    return df

deterministic_df = read_csv_or_empty('deterministic.csv')
categorical_df = read_csv_or_empty('categorical.csv')
probabilistic_df = read_csv_or_empty('probabilistic.csv')
fss_df = read_csv_or_empty('fss.csv')
rank_df = read_csv_or_empty('rank_histograms.csv')
reliability_df = read_csv_or_empty('reliability.csv')
histogram_df = read_csv_or_empty('histograms.csv')
summary_df = read_csv_or_empty('summary_main_metrics.csv')

if not summary_df.empty:
    display(summary_df)

## Main deterministic metrics

These curves use the deterministic forecast for deterministic models and the ensemble mean for probabilistic models.

In [ ]:
def plot_metric_by_lead(df, metric, ylabel, title, filename, query=None):
    if df.empty or metric not in df.columns:
        print('No data for', metric)
        return
    sub = df.copy()
    if query:
        sub = sub.query(query).copy()
    if sub.empty:
        print('No data after filtering for', metric)
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    for label, g in sub.groupby('model_label'):
        curve = g.groupby('lead_time_min', as_index=False)[metric].mean().sort_values('lead_time_min')
        ax.plot(curve['lead_time_min'], curve[metric], marker='o', label=label)
    ax.set_xlabel('Lead time (minutes)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    out = FIG_DIR / filename
    fig.savefig(out, dpi=180)
    plt.show()
    print('Saved', out)

plot_metric_by_lead(deterministic_df, 'rmse', 'RMSE (mm/h)', 'RMSE by lead time', 'rmse_by_lead.png')
plot_metric_by_lead(deterministic_df, 'me', 'Mean error / bias (mm/h)', 'Mean error by lead time', 'me_by_lead.png')
plot_metric_by_lead(deterministic_df, 'mae', 'MAE (mm/h)', 'MAE by lead time', 'mae_by_lead.png')
plot_metric_by_lead(deterministic_df, 'mape_obs_ge_0p1', 'MAPE for observed rain ≥ 0.1 mm/h', 'MAPE by lead time', 'mape_by_lead.png')

## Threshold-based categorical metrics

In [ ]:
for q in THRESHOLDS_MM_H:
    plot_metric_by_lead(
        categorical_df,
        'csi',
        f'CSI at {q:g} mm/h',
        f'CSI by lead time, threshold {q:g} mm/h',
        f'csi_{str(q).replace(".", "p")}_by_lead.png',
        query=f'threshold_mm_h == {q}',
    )

if not categorical_df.empty:
    cat_summary = (
        categorical_df
        .groupby(['model_label', 'threshold_mm_h'], as_index=False)
        .agg(csi=('csi', 'mean'), pod=('pod', 'mean'), far=('far', 'mean'), ets=('ets', 'mean'))
        .sort_values(['threshold_mm_h', 'csi'], ascending=[True, False])
    )
    display(cat_summary)
    cat_summary.to_csv(METRICS_DIR / 'categorical_summary_by_threshold.csv', index=False)

## Probabilistic metrics

In [ ]:
if not probabilistic_df.empty:
    crps_df = probabilistic_df[probabilistic_df['metric'] == 'crps'].copy()
    crps_df['crps'] = crps_df['value']
    plot_metric_by_lead(crps_df, 'crps', 'CRPS (mm/h)', 'CRPS by lead time', 'crps_by_lead.png')

    for q in THRESHOLDS_MM_H:
        brier_df = probabilistic_df[(probabilistic_df['metric'] == 'brier') & (probabilistic_df['threshold_mm_h'] == q)].copy()
        brier_df['brier'] = brier_df['value']
        plot_metric_by_lead(brier_df, 'brier', f'Brier score at {q:g} mm/h', f'Brier score by lead time, threshold {q:g} mm/h', f'brier_{str(q).replace(".", "p")}_by_lead.png')
else:
    print('No probabilistic metrics loaded.')

## Fractions skill score

In [ ]:
if not fss_df.empty:
    for q in [1.0, 5.0]:
        for s in [10, 30, 60]:
            plot_metric_by_lead(
                fss_df,
                'fss',
                f'FSS q={q:g} mm/h, window={s}px',
                f'FSS by lead time, q={q:g} mm/h, window={s}px',
                f'fss_q{str(q).replace(".", "p")}_w{s}_by_lead.png',
                query=f'threshold_mm_h == {q} and window_size_px == {s}',
            )
else:
    print('No FSS metrics loaded.')

## Rank histograms and reliability summaries

In [ ]:
if not rank_df.empty:
    for model_label, g in rank_df.groupby('model_label'):
        counts = g.groupby('rank_bin', as_index=False)['count'].sum().sort_values('rank_bin')
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(counts['rank_bin'], counts['count'])
        ax.set_xlabel('Observation rank among ensemble members')
        ax.set_ylabel('Count')
        ax.set_title(f'Rank histogram | {model_label}')
        fig.tight_layout()
        out = FIG_DIR / f'rank_hist_{model_label.lower().replace(" ", "_").replace("-", "_")}.png'
        fig.savefig(out, dpi=180)
        plt.show()
        print('Saved', out)

if not reliability_df.empty:
    rel_summary = (
        reliability_df
        .groupby(['model_label', 'threshold_mm_h', 'prob_bin_left', 'prob_bin_right'], as_index=False)
        .agg(n=('n', 'sum'), observed_frequency=('observed_frequency', 'mean'))
    )
    display(rel_summary.head())
    rel_summary.to_csv(METRICS_DIR / 'reliability_summary.csv', index=False)

## Final result files

In [ ]:
print('Figures saved in:', FIG_DIR)
print('Metrics saved in:', METRICS_DIR)
for p in sorted(METRICS_DIR.glob('*.csv')):
    print('-', p.name)